# STAR — Kaggle V5: SO SANH best.pth (cua ban) vs X-VLM GOC tren old-test THAT

**Khong dung distractor** — gallery = dung **1.978 anh that** cua old-test (protocol giong paper SOTA).
Moi anh vua la **query** vua la **ung vien gallery**; GT cua moi query = chinh anh do.

**So sanh 2 model x 3 tang:**
- `trained` = best.pth ban vua train (auto-chon ckpt best_metric cao nhat). Neu no train POSE-ON va co
  file ViTPose cho old-test → **eval CO pose** (cong bang voi luc train). Khong co json → pose-OFF (canh bao).
- `base`    = X-VLM 16M GOC (chua finetune, **luon pose-OFF**) — de biet finetune GIUP hay HAI transfer that.
- 3 tang: **stage1** (cosine, KHONG cross) → **rerank** (+ITM cross-encoder) → **gale_shapley** (+1-1).

**Pose hoat dong the nao:** pipeline fuse pose vao dac trung ANH (ca 1.978 anh deu o gallery), KHONG phai text.
Cross-encoder (rerank) KHONG dung pose → tang do luon cong bang ca 2 model.

**Cai dat:** GPU T4 · Internet ON · **Add Input:**
1. `star-oldtest` (`attr.json` + folder `test/` 1.978 jpg)
2. `star-10k-hard` (co `xvlm_16m_base.th`) — hoac de notebook gdown
3. dataset chua **`best.pth`** ban vua train
4. *(tuy chon, de eval trained CO pose)* dataset chua **`*vitpose*.json`** cho 1.978 anh old-test
   (cung format `train_10k_hard_vitpose.json`: items[image_id].instances[0].keypoints_xyc + width/height)

**Thoi gian:** setup ~10' + ~8-12'/(model×mode) → mac dinh 2 model × 2 mode ≈ **~45-60'**.

In [ ]:
# [1/8] GPU + clone/pull repo (can ban MOI co src/star/inference + pose-in-pipeline)
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4!"
print("GPU:", gpu)
os.chdir("/kaggle/working")
if pathlib.Path("star/.git").exists():
    !cd star && git pull -q
else:
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
src = pathlib.Path("src/star/inference/pipeline.py").read_text()
assert "keypoints" in src, "repo cu — chua co pose-in-pipeline, pull lai!"
print("repo OK (inference + pose):", os.getcwd())

In [ ]:
# [2/8] Env pinned + X-VLM + 4 patches
!python scripts/kaggle_setup.py

In [ ]:
# [3/8] Tim inputs: old-test, X-VLM base, best.pth (chon metric cao nhat), va (tuy chon) ViTPose json
import glob, pathlib, torch

def find_all(p):
    return sorted(set(glob.glob(f"/kaggle/input/*/{p}") + glob.glob(f"/kaggle/input/*/**/{p}", recursive=True)))

attr = find_all("attr.json")
assert attr, "Khong thay attr.json — Add Input dataset star-oldtest!"
ATTR = attr[0]; OLD_ROOT = str(pathlib.Path(ATTR).parent)
probe = find_all("test/0.jpg") or find_all("0.jpg")
assert probe, "Khong thay anh test/0.jpg trong dataset old-test"
if not pathlib.Path(OLD_ROOT, "test/0.jpg").exists():
    OLD_ROOT = str(pathlib.Path(probe[0]).parents[1])

base = find_all("xvlm_16m_base.th")
if base:
    BASE_CKPT = base[0]
else:
    import os
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    BASE_CKPT = "data/checkpoints/xvlm_16m_base.th"
assert pathlib.Path(BASE_CKPT).stat().st_size > 8e8

cands = find_all("best.pth")
assert cands, "Khong thay best.pth nao — Add Input dataset chua best.pth ban da train!"
print("Tim thay best.pth:")
scored = []
for c in cands:
    try:
        raw = torch.load(c, map_location="cpu", weights_only=False)
        bm = raw.get("best_metric")
        pe = ((raw.get("extra") or {}).get("cfg") or {}).get("model", {}).get("pose_enabled")
        gb = ((raw.get("extra") or {}).get("cfg") or {}).get("data", {}).get("group_by")
        del raw
    except Exception as e:
        bm, pe, gb = None, "?", f"(loi: {e})"
    scored.append((bm if bm is not None else -1, c, pe, gb))
    print(f"  {c} | best_metric={bm} pose_enabled={pe} group_by={gb}")
scored.sort(reverse=True)
_, BEST_CKPT, BEST_POSE, _ = scored[0]

# ViTPose json cho old-test (tuy chon) — chi can neu BEST_POSE=true muon eval CO pose
VITPOSE = (find_all("*vitpose*.json") or [None])[0]
print(f"\n=> CHON: {BEST_CKPT} (best_metric={scored[0][0]}, pose_enabled={BEST_POSE})")
print("OLD_ROOT  =", OLD_ROOT)
print("BASE_CKPT =", BASE_CKPT)
print("VITPOSE   =", VITPOSE or "(khong co -> trained se eval POSE-OFF, lech train/eval)")

In [ ]:
# [4/8] Build manifest gallery-1978 (KHONG distractor) + keypoints (neu co ViTPose)
import json
import pandas as pd

MODES = ["human", "vlm"]   # human=source_caption (ngan) | vlm=caption chi tiet (giong train)

rows = [json.loads(l) for l in open(ATTR, encoding="utf-8")]
assert len(rows) == 1978, f"attr.json co {len(rows)} dong (mong doi 1978)"

pose_items = {}
if VITPOSE:
    pose_items = json.load(open(VITPOSE, encoding="utf-8")).get("items", {})
def kpts_of(iid):
    it = pose_items.get(str(iid))
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return [0.0] * 51 if VITPOSE else None      # zero-fill khi co json nhung anh nay thieu kpts
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else ([0.0] * 51 if VITPOSE else None)

def cap_of(r, mode):
    return (r.get("source_caption") or r["caption"])[0] if mode == "human" else r["caption"][0]

MANIFESTS = {}
for mode in MODES:
    out = [dict(image_path=r["image"], caption=cap_of(r, mode), split="valb",
                sequence_id=f'o{r["image_id"]}', scene=f'o{r["image_id"]}', action="q",
                image_id=str(r["image_id"]), bbox=None, keypoints=kpts_of(r["image_id"])) for r in rows]
    df = pd.DataFrame(out)
    assert df.image_id.nunique() == 1978 and (df.caption != "").sum() == 1978   # tat ca la query
    mp = f"/kaggle/working/oldtest_{mode}.parquet"; df.to_parquet(mp, index=False); MANIFESTS[mode] = mp
KCOV = pd.read_parquet(MANIFESTS["human"]).keypoints.notna().mean() if VITPOSE else 0.0
print(f"manifests {MODES} | 1978 query=gallery (khong distractor) | kpts coverage: {KCOV:.0%}")
print("vd human:", pd.read_parquet(MANIFESTS['human']).caption.iloc[0][:70])

In [ ]:
# [5/8] Helper: dung model (trained / base) + chay pipeline 3 tang
import sys, torch
sys.path.insert(0, "src")
from star.config import load_config, _merge
from star.data import PABDataset
from star.inference import run_pipeline
from star.models import STARModel

def build_trained():
    raw = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
    cfg = load_config("configs/star_v3_10k_kaggle.yaml")
    emb = (raw.get("extra") or {}).get("cfg") or {}
    if "model" in emb:
        _merge(cfg.model, emb["model"])          # khoi phuc DUNG kien truc (lora_r, pose_enabled,...)
    cfg.model.checkpoint = BASE_CKPT
    model = STARModel(cfg).to("cuda").eval()
    msg = model.load_state_dict(raw["model"], strict=False)
    pose_active = (model.pose is not None) and bool(VITPOSE)
    print(f"  trained: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)} "
          f"| pose branch={'co' if model.pose is not None else 'khong'} | "
          f"POSE {'ON (dung ViTPose)' if pose_active else 'OFF'}")
    if model.pose is not None and not VITPOSE:
        print("  ⚠️  ckpt train POSE-ON nhung KHONG co ViTPose json -> stage1 bi lech train/eval (rerank van cong bang)")
    del raw
    return model, cfg

def build_base():
    cfg = load_config("configs/star_v3_10k_kaggle.yaml")
    cfg.model.checkpoint = BASE_CKPT
    cfg.model.lora_enabled = False     # X-VLM GOC: khong LoRA
    cfg.model.pose_enabled = False     # luon pose-OFF
    model = STARModel(cfg).to("cuda").eval()
    print("  base: X-VLM goc (khong LoRA, khong pose)")
    return model, cfg

print("helper san sang")

In [ ]:
# [6/8] CHAY SO SANH: {trained, base} x MODES x {stage1, rerank, gale_shapley}
import json, pathlib, time, torch
RESULTS = []
for model_name, builder in [("trained", build_trained), ("base_xvlm", build_base)]:
    print("=" * 90); print(">>>", model_name)
    model, cfg = builder()
    for mode in MODES:
        ds = PABDataset(MANIFESTS[mode], OLD_ROOT, model.backbone.tokenizer, split="valb",
                        image_size=cfg.data.image_size, max_token=cfg.data.max_token, train=False)
        t0 = time.time()
        res = run_pipeline(model, ds, "cuda", topk=100, batch_size=64, num_workers=2)
        mins = (time.time() - t0) / 60
        for stage in ("stage1", "rerank", "gale_shapley"):
            RESULTS.append(dict(model=model_name, mode=mode, stage=stage,
                                **{k: round(v, 4) for k, v in res[stage].items()}))
        out = pathlib.Path(f"/kaggle/working/eval_v5/{model_name}_{mode}"); out.mkdir(parents=True, exist_ok=True)
        (out / "metrics.json").write_text(json.dumps(
            {s: res[s] for s in ("stage1", "rerank", "gale_shapley")} |
            {"gallery": res["gallery_size"], "queries": res["num_queries"]}, indent=2))
        print(f"  [{mode:5s}] stage1 mAP={res['stage1']['mAP']:.4f}  rerank={res['rerank']['mAP']:.4f}  "
              f"GS={res['gale_shapley']['mAP']:.4f}  ({mins:.1f}')")
    del model; torch.cuda.empty_cache()

In [ ]:
# [7/8] BANG TONG HOP + delta trained-vs-base
import json, pathlib
import pandas as pd
tab = pd.DataFrame(RESULTS)
print(tab.to_string(index=False))
print("\n================ mAP: trained vs base (delta = trained - base) ================")
m = tab.pivot_table(index=["mode", "stage"], columns="model", values="mAP")
m["delta"] = m["trained"] - m["base_xvlm"]
print(m.to_string())
pathlib.Path("/kaggle/working/eval_v5/summary.json").write_text(json.dumps(RESULTS, indent=2))
print("\nLuu: /kaggle/working/eval_v5/<model>_<mode>/metrics.json + summary.json")

## [8/8] Doc ket qua

**Cau hoi 1 — finetune GIUP hay HAI? (cot `delta`)**
- `delta > 0` (trained > base): finetune synthetic CO giup transfer that → day manh huong v3c.
- `delta < 0` (base > trained): finetune **PHA** bieu dien anh-that → overfit synthetic → doi chien luoc.

**Cau hoi 2 — re-rank dong gop bao nhieu? (stage1 → rerank)** — cross-encoder khong dung pose → cong bang ca 2.

**Cau hoi 3 — caption ngan vs dai? (`human` vs `vlm`)** — `vlm` cao hon nhieu → caption ngan (de nam nay) se kho → can augment.

**Pose:** neu co ViTPose json, `trained` eval CO pose (khop luc train, cong bang). Neu khong → `trained` stage1
bi phat (lech train/eval), nhin `rerank` de danh gia dung hon. `base` luon pose-OFF.

**Nhac luat:** test (masked) nam nay KHONG duoc dung lam distractor/validation duoi moi hinh thuc.